# 🚀 Laya GPU Curation Pipeline (Google Colab / Kaggle)

This notebook runs the **Laya Non-Autoregressive Decision Engine**
- Evaluates all **8,058 conversation threads**.
- Outputs: `rag_clean.parquet` (Gold Curated Dataset) & `data_curation_report.json`.

### Step 1: Check GPU Acceleration
Ensure your Colab runtime is set to **T4 GPU** (*Runtime -> Change runtime type -> T4 GPU*).

In [ ]:
!nvidia-smi

### Step 2: Install Dependencies
Installs `laya`, `polars`, `pyarrow`, and `torch`.

In [ ]:
!pip install --upgrade laya polars pyarrow tqdm

### Step 3: Upload `cleaned_threads.parquet` (29.6 MB)
Upload the file from your local `*/cleaned_threads.parquet`.

In [ ]:
import os
from pathlib import Path

input_file = Path("cleaned_threads.parquet")

if not input_file.exists():
    try:
        from google.colab import files
        print("Please select and upload 'cleaned_threads.parquet' from your local data/interim/ folder:")
        uploaded = files.upload()
    except ImportError:
        print("On Kaggle: Add 'cleaned_threads.parquet' to Kaggle input dataset.")
else:
    print(f"Found existing '{input_file}' ({input_file.stat().st_size / (1024*1024):.1f} MB)")

### Step 4: Run High-Speed GPU Curation with Laya

In [ ]:
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List

import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from tqdm import tqdm
import laya

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Inference Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Fast pre-filter for promo/link posts
META_PROMO_PATTERNS = [
    r"\bdiscord(?:\.gg|\.com|\s+server)?\b",
    r"\btelegram\b",
    r"\bwhatsapp\b",
    r"\bjoin\s+(?:our|the)\s+(?:discord|server|community|group)\b",
    r"\bsubreddit\s+(?:rules|update|moderator|feedback)\b",
    r"\bwelcome\s+to\s+(?:the\s+)?(?:sub|community|rag)\b",
    r"\bweekly\s+discussion\s+thread\b",
    r"\bmonthly\s+recap\b",
]

def is_bare_url_or_promo(title: str, body: str) -> bool:
    combined = f"{title} {body}".lower().strip()
    for pat in META_PROMO_PATTERNS:
        if re.search(pat, combined):
            return True
    stripped = body.strip()
    if stripped.startswith("http://") or stripped.startswith("https://"):
        if len(stripped.split()) <= 10:
            return True
    if len(stripped) < 100 and ("http://" in stripped or "https://" in stripped):
        words = [w for w in stripped.split() if not w.startswith("http")]
        if len(words) < 6:
            return True
    return False

RAG_CURATION_QUESTIONS = {
    "is_rag_related": {
        "type": "noul",
        "instructions": (
            "Is this question or discussion specifically about Retrieval-Augmented Generation (RAG), "
            "vector databases, embeddings, chunking, semantic or hybrid search, context retrieval, "
            "reranking, or LLM grounding?"
        ),
    },
    "technical_quality": {
        "type": "score",
        "instructions": "Rate the technical depth, quality, and usefulness of this discussion.",
        "criteria": [
            "low: vague, beginner rant, promotional, superficial, or low technical detail",
            "medium: standard practical question, bug troubleshooting, or working setup advice",
            "high: deep technical architecture, benchmark, detailed diagnosis, or working production solution",
        ],
    },
    "taxonomy_topic": {
        "type": "choice",
        "instructions": "What is the primary technical domain of this discussion?",
        "criteria": {
            "retrieval": "vector search, hybrid search, BM25, reranking, cross-encoders, query expansion",
            "chunking_and_indexing": "chunking strategies, embeddings, metadata filtering, vector databases like Qdrant/Chroma/Milvus",
            "generation_and_grounding": "prompt engineering, context grounding, hallucination mitigation, citations",
            "evaluation_and_benchmarks": "faithfulness, answer relevance, recall, precision, Ragas, TruLens, benchmarks",
            "agentic_and_graph_rag": "agentic workflows, knowledge graphs, corrective RAG (CRAG), multi-hop reasoning",
            "tooling_and_frameworks": "LangChain, LlamaIndex, Ollama, frameworks, configuration, local deployment",
        },
    },
}

# Load Laya Model (Fine-tuned typed-decisions checkpoint)
print("Loading Laya checkpoint on GPU...")
t_load = time.time()
try:
    agent = laya.load("convaiinnovations/laya", subfolder="typed-decisions", device=device)
except Exception as e:
    print(f"Falling back to root checkpoint: {e}")
    agent = laya.load("convaiinnovations/laya", device=device)
print(f"Loaded in {time.time() - t_load:.1f}s")

# Load input data
df = pl.read_parquet("cleaned_threads.parquet")
rows = df.to_dicts()
total = len(rows)
print(f"Total threads to evaluate: {total:,}")

curated_records = []
rejected_records = []
topic_counts = {}
latencies = []
pre_filtered_promos = 0

RAG_THRESHOLD = 0.50      # Calibrated cutoff (allows genuine technical Qs)
MIN_QUALITY_SCORE = 0.60  # Minimum technical substance

t_start = time.time()
for row in tqdm(rows, desc="GPU Curating"): 
    title = row.get("title", "")
    post_body = row.get("post_body", "")

    # 1. Pre-filter promo/bare links
    if is_bare_url_or_promo(title, post_body):
        pre_filtered_promos += 1
        entry = dict(row)
        entry.update({
            "is_rag_related": 0.05,
            "technical_quality_score": 0.1,
            "taxonomy_topic": "meta_or_promo",
            "topic_confidence": 0.99,
            "curation_passed": False,
        })
        rejected_records.append(entry)
        continue

    # 2. Format State within token budget
    state = {"title": title}
    if post_body:
        state["post_body"] = post_body[:600]
    best_ans = str(row.get("best_answer_body") or "").strip()
    if best_ans:
        state["best_answer"] = best_ans[:600]

    # 3. Laya Forward Pass
    t0 = time.perf_counter()
    res = agent.predict(state, RAG_CURATION_QUESTIONS)
    dt_ms = (time.perf_counter() - t0) * 1000.0
    latencies.append(dt_ms)

    answers = res.get("answers", {})
    rag_prob = float(answers.get("is_rag_related", {}).get("noul", 0.0))
    qual_score = float(answers.get("technical_quality", {}).get("score", 0.0))
    topic_ans = answers.get("taxonomy_topic", {})
    topic_choice = str(topic_ans.get("choice", "other"))
    topic_conf = float(topic_ans.get("confidence", 0.0))

    topic_counts[topic_choice] = topic_counts.get(topic_choice, 0) + 1

    passed = bool(rag_prob >= RAG_THRESHOLD and qual_score >= MIN_QUALITY_SCORE)

    entry = dict(row)
    entry.update({
        "is_rag_related": round(rag_prob, 4),
        "technical_quality_score": round(qual_score, 4),
        "taxonomy_topic": topic_choice,
        "topic_confidence": round(topic_conf, 4),
        "curation_passed": passed,
    })

    if passed:
        curated_records.append(entry)
    else:
        rejected_records.append(entry)

total_sec = time.time() - t_start
avg_ms = sum(latencies) / len(latencies) if latencies else 0.0
pass_rate = len(curated_records) / total * 100 if total else 0

print(f"\n=== COMPLETED IN {total_sec/60:.2f} MINUTES! ===")
print(f"Total Evaluated: {total:,}")
print(f"Pre-filtered Promos/Links: {pre_filtered_promos:,}")
print(f"Curated Gold Passed: {len(curated_records):,} ({pass_rate:.1f}%)")
print(f"Average Latency: {avg_ms:.1f} ms/thread")
print(f"Taxonomy Breakdown: {json.dumps(topic_counts, indent=2)}")

# Save Parquet Files
curated_df = pl.DataFrame(curated_records)
curated_df.write_parquet("rag_clean.parquet", compression="zstd")
print(f"Saved 'rag_clean.parquet' ({Path('rag_clean.parquet').stat().st_size / (1024*1024):.2f} MB)")

report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "total_evaluated": total,
    "pre_filtered_promos": pre_filtered_promos,
    "curated_passed": len(curated_records),
    "curated_rejected": len(rejected_records),
    "pass_rate_pct": round(pass_rate, 2),
    "avg_latency_ms": round(avg_ms, 2),
    "total_runtime_sec": round(total_sec, 2),
    "taxonomy_distribution": topic_counts,
}
with open("data_curation_report.json", "w") as f:
    json.dump(report, f, indent=2)
print("Saved 'data_curation_report.json'")

### Step 5: Download Curated Dataset Back to Local Machine
Download `rag_clean.parquet` and place it in your local `data/processed/` folder!

In [ ]:
try:
    from google.colab import files
    print("Downloading 'rag_clean.parquet'...")
    files.download("rag_clean.parquet")
    files.download("data_curation_report.json")
except ImportError:
    print("On Kaggle: Find output files in /kaggle/working/ and download directly.")